# AUSA attorney tracker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abigailhaddad/ausa-attorney-tracker/blob/main/ausa_attorney_tracker.ipynb)

Monthly headcount, hiring, and departures for Assistant U.S. Attorneys (DOJ, Executive Office for U.S. Attorneys and the Offices of the U.S. Attorneys, occupational series 0905), queried **live** from the public OPM/EHRI mirror on HuggingFace ([`impactproject/opm-ehri-data`](https://huggingface.co/datasets/impactproject/opm-ehri-data)) with DuckDB over HTTPS. Nothing is downloaded to disk.

**Scope: DC vs. rest-of-country, not state-by-state.** Every geographic field in this data (`duty_station_state_abbreviation`, `duty_station_city`, `core_based_statistical_area`) is privacy-redacted for ~91% of AUSA records — a small-occupational-subgroup suppression rule applied uniformly across the whole location hierarchy. DC is the one exception, since its ~500-attorney cell is large enough to clear the suppression threshold. So the only two honest "area" buckets available are **DC** and **rest-of-country (aggregate)** — this notebook does not fabricate state-level detail the data doesn't actually contain.

In [ ]:
# Setup — installs duckdb/pandas/great_tables if missing (all pip-installable on Colab)
for pkg, mod in [("duckdb", "duckdb"), ("pandas", "pandas"), ("great_tables", "great_tables")]:
    try:
        __import__(mod)
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import duckdb
import pandas as pd
from great_tables import GT
import urllib.request
import json
import re

In [ ]:
# Config
REPO = "impactproject/opm-ehri-data"
HF = f"https://huggingface.co/datasets/{REPO}/resolve/main/"
AGENCY_SUBELEMENT = "EXECUTIVE OFFICE FOR U.S. ATTORNEYS AND THE OFFICES OF THE U.S. ATTORNEYS"
SERIES_CODE = "0905"  # attorney

# Nov 2024 (pre-inauguration baseline) through present. Accessions/separations
# are small files (KiB-low-MiB) and stay monthly across this whole window.
# Employment snapshots are the full federal workforce each month (26-75 MiB
# per file) filtered down to ~6,000 AUSA rows -- sampled at an interval
# instead of pulled monthly, both to keep the notebook fast and to avoid
# tripping HuggingFace's per-window rate limit on repeated large-file reads.
START_YM = "202411"
END_YM = "202612"
EMPLOYMENT_SAMPLE_STRIDE = 3  # every 3rd available employment month (~quarterly)

# appointment_type values that mean "political appointee," excluded below so
# the tables track the career AUSA workforce. Confirmed by enumerating every
# distinct appointment_type actually present for this population Nov 2024-
# present (2026-08-08) and checking each one, not guessed:
#   - SCHEDULE C: the standard political-appointee schedule (5 CFR 213.3301)
#   - NONCAREER (SENIOR EXECUTIVE SERVICE PERMANENT): noncareer SES = political
#   - EXECUTIVE (EXCEPTED SERVICE NONPERMANENT): always pay_plan_code='AD',
#     grade='40', supervisory_status='SUPERVISOR OR MANAGER' -- one specific,
#     consistent combination, consistent with the (Presidentially-appointed,
#     Senate-confirmed) U.S. Attorney / top leadership slot per district, not
#     an ordinary career attorney classification.
# NOTE: "OTHER (EXCEPTED SERVICE NONPERMANENT)" is NOT excluded even though
# "NONPERMANENT" sounds temporary -- checked back to 2015 and it's the
# dominant code for ordinary AUSA hires in every year, not a marker of
# temporary/surge staffing specific to this window.
POLITICAL_APPOINTMENT_TYPES = [
    "SCHEDULE C (EXCEPTED SERVICE NONPERMANENT)",
    "NONCAREER (SENIOR EXECUTIVE SERVICE PERMANENT)",
    "EXECUTIVE (EXCEPTED SERVICE NONPERMANENT)",
]

In [ ]:
def list_all_files(repo=REPO):
    """Every file in the HF tree, following pagination.

    The tree API caps a single response at 1000 entries (Link header,
    rel="next", cursor-based). This repo already has 1000+ files across
    accessions/employment/separations combined — a one-shot fetch silently
    truncates whichever directory sorts last alphabetically (separations/).
    """
    url = f"https://huggingface.co/api/datasets/{repo}/tree/main?recursive=true&limit=1000"
    out = []
    while url:
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req) as r:
            headers = dict(r.getheaders())
            out.extend(json.load(r))
        link = headers.get("Link")
        m = re.search(r'<([^>]+)>;\s*rel="next"', link) if link else None
        url = m.group(1) if m else None
    return out


def monthly_urls(files, dataset, start, end):
    """Latest version per month, from `start` through `end` (YYYYMM strings)."""
    best = {}
    for f in files:
        m = re.search(dataset + r"_(\d{6})_v(\d+)\.parquet", f["path"])
        if not m:
            continue
        month, ver = m.group(1), int(m.group(2))
        if start <= month <= end and (month not in best or ver > best[month][0]):
            best[month] = (ver, f["path"])
    return [HF + best[m][1] for m in sorted(best)]


files = list_all_files()
print(f"{len(files)} files total in the HF repo")

In [ ]:
con = duckdb.connect()
con.execute("SET enable_progress_bar=false;")
con.execute("INSTALL httpfs; LOAD httpfs;")
# Safety net, not the primary defense — the real fix against HuggingFace's
# rate limit is querying far fewer/smaller files in the first place (see
# START_YM/EMPLOYMENT_SAMPLE_STRIDE above). If a cell still errors with
# HTTP 429 after all retries, just re-run it — it's a temporary throttle.
con.execute("SET http_retries=6;")
con.execute("SET http_retry_wait_ms=1000;")
con.execute("SET http_retry_backoff=2;")
con.execute("SET threads=4;")

AREA_CASE = "CASE WHEN duty_station_state_abbreviation='DC' THEN 'DC' ELSE 'Rest of country' END"
POLITICAL_EXCLUSION_SQL = "(" + ",".join(f"'{t}'" for t in POLITICAL_APPOINTMENT_TYPES) + ")"


def query_monthly(dataset, date_col, value_label, stride=1):
    """Monthly (ym, area, value_label) totals for the career AUSA population,
    one dataset at a time.

    `stride` samples every Nth available file instead of every one -- only
    used for employment, whose files are far larger than accessions'/
    separations'. Filters on `{date_col} BETWEEN START_YM AND END_YM`
    explicitly, not just on which files get fetched: a file named for one
    month can contain a handful of records with an older effective date (a
    late-processed correction), and those would otherwise leak in as stray
    columns outside the window this notebook is about. Also excludes
    POLITICAL_APPOINTMENT_TYPES (see config cell) so hiring/departure/
    headcount figures reflect the career AUSA workforce, not political
    leadership turnover.
    """
    urls = monthly_urls(files, dataset, START_YM, END_YM)[::stride]
    lst = "[" + ",".join(f"'{u}'" for u in urls) + "]"
    return con.execute(f"""
        SELECT {date_col} AS ym,
               {AREA_CASE} AS area,
               SUM(TRY_CAST(count AS BIGINT)) AS {value_label}
        FROM read_parquet({lst}, union_by_name=true)
        WHERE agency_subelement = '{AGENCY_SUBELEMENT}'
          AND occupational_series_code = '{SERIES_CODE}'
          AND {date_col} BETWEEN '{START_YM}' AND '{END_YM}'
          AND appointment_type NOT IN {POLITICAL_EXCLUSION_SQL}
        GROUP BY 1, 2
    """).df()

## Query the three cubes

Accessions and separations pull every available month Nov 2024–present.
Employment is sampled every `EMPLOYMENT_SAMPLE_STRIDE`-th available month
(~quarterly) instead of monthly, since those snapshot files are the full
federal workforce (26–75 MiB each) filtered down to ~6,000 AUSA rows.

In [ ]:
print("Querying accessions (hires)...")
df_accessions = query_monthly("accessions", "personnel_action_effective_date_yyyymm", "hires")
print(f"  {len(df_accessions)} (month, area) rows, {df_accessions.ym.nunique()} distinct months")

print("Querying separations...")
df_separations = query_monthly("separations", "personnel_action_effective_date_yyyymm", "separations")
print(f"  {len(df_separations)} (month, area) rows, {df_separations.ym.nunique()} distinct months")

print(f"Querying employment (headcount), every {EMPLOYMENT_SAMPLE_STRIDE}rd available month...")
df_employment = query_monthly("employment", "snapshot_yyyymm", "headcount", stride=EMPLOYMENT_SAMPLE_STRIDE)
print(f"  {len(df_employment)} (month, area) rows, {df_employment.ym.nunique()} distinct months")

## Tables

[great_tables](https://posit-dev.github.io/great-tables/) instead of a
matplotlib heatmap — a color-shaded table reads more clearly than an
`imshow` grid at this size, with real numbers in every cell instead of tiny
rotated-axis labels. One row per month, DC / rest-of-country / **Total**
(their sum, so nationwide totals are never a mental-math exercise).

Color is used consistently across all four tables, not just within each
one: **red always means bad, blue always means good**, never magnitude for
its own sake. Hires are colored blue (more = better); separations are
colored red (more = worse) — using the *same* yellow-orange "more = darker"
scale for both would have told two contradictory stories with identical
colors. Net hires and the headcount % columns use a diverging red
(loss) / blue (gain) scale centered on zero (net) or on the baseline
(headcount).

In [ ]:
GOOD_PALETTE = ["#F7FBFF", "#6BAED6", "#2166AC"]   # light -> dark blue: higher = more hires = better
BAD_PALETTE = ["#FFF5F0", "#FB6A4A", "#B2182B"]     # light -> dark red: higher = more separations = worse
DIVERGING_PALETTE = ["#B2182B", "#F7F7F7", "#2166AC"]  # red (loss/below baseline) - white - blue (gain/above baseline)
ATTY_NOTE = "career attorneys only, series 0905 — excludes political appointees"
# Every table gets this so a screenshot of just ONE table, with no
# surrounding notebook context, still says where the data is from and
# what "DC / rest-of-country" does and doesn't mean.
SOURCE_NOTE = (
    "Source: OPM/EHRI (impactproject/opm-ehri-data on HuggingFace), queried live. "
    "DC vs. rest-of-country only — finer geography (state/city) is privacy-redacted "
    "for this population."
)

# Fixed, generous domains for the two "% of a baseline" columns below -- NOT
# derived from this window's own min/max. A domain scaled to just this
# window's own extremes makes the worst month always look maximally
# saturated no matter how mild the real swing is (a 9% decline looked
# nearly as dark red as a 14% one when the domain was [85,100]). Anchoring
# to a fixed, meaningfully-extreme reference instead means color intensity
# reflects how bad things actually are, not just "worst cell inside
# whatever window I happen to be looking at right now."
HEADCOUNT_PCT_DOMAIN = [75, 125]   # full color at a +/-25pp swing from baseline headcount
NET_PCT_DOMAIN = [-15, 15]         # full color at a net change of 15% of headcount in one month


def to_area_table(df, value_col):
    """One row per month, DC / rest-of-country / Total (their sum) as columns."""
    pivot = df.pivot_table(index="ym", columns="area", values=value_col, aggfunc="sum")
    for c in ("DC", "Rest of country"):
        if c not in pivot.columns:
            pivot[c] = pd.NA
    pivot = pivot[["DC", "Rest of country"]].sort_index().reset_index().rename(columns={"ym": "Month"})
    pivot["Total"] = pivot["DC"].fillna(0) + pivot["Rest of country"].fillna(0)
    return pivot


def build_gt(df, value_col, title, mode="good"):
    """One row per month, DC/rest-of-country/Total as separately-colored columns.

    mode="good": higher is better (hires) -> blue, darker = more.
    mode="bad": higher is worse (separations) -> red, darker = more.
    mode="diverging": below zero is worse (net) -> red/white/blue centered on 0.
    Deliberately NOT a single "magnitude" palette reused for both hires and
    separations -- more hires is good news and more separations is bad
    news, so they need visually opposite colors, not the same color at a
    different position. Raw-count domains here ARE data-derived (unlike
    the two % columns below) because the darkest cell in each is a
    genuinely real standout event (e.g. Sep 2025 separations), not an
    artifact of a narrow window.
    """
    tbl = to_area_table(df, value_col)
    cols = ["DC", "Rest of country", "Total"]
    gt = (
        GT(tbl)
        .tab_header(title=f"{title} — monthly, {ATTY_NOTE}")
        .tab_source_note(SOURCE_NOTE)
        .fmt_integer(columns=cols)
        .sub_missing(columns=cols, missing_text="—")
    )
    for col in cols:
        vals = tbl[col].dropna()
        if vals.empty:
            continue
        if mode == "diverging":
            vmax = float(vals.abs().max() or 1)
            gt = gt.data_color(columns=[col], palette=DIVERGING_PALETTE, domain=[-vmax, vmax])
        elif mode == "bad":
            gt = gt.data_color(columns=[col], palette=BAD_PALETTE, domain=[float(vals.min()), float(vals.max())])
        else:
            gt = gt.data_color(columns=[col], palette=GOOD_PALETTE, domain=[float(vals.min()), float(vals.max())])
    return gt


def build_employment_gt(df):
    """Headcount table. Raw counts (DC/rest-of-country/Total) are left
    UNCOLORED on purpose -- coloring them on their own scale while the %
    columns use a diverging scale told two contradictory stories with the
    same dark color (dark = high count = good, vs. dark = far from
    baseline = bad), right next to each other in the same row. Only the %
    columns carry color, all three sharing ONE fixed domain
    (HEADCOUNT_PCT_DOMAIN), so the same percentage always renders as the
    same shade regardless of which area column it's in, DC's shade is
    directly comparable to rest-of-country's, and the color intensity
    means the same thing across different runs of this notebook.
    """
    tbl = to_area_table(df, "headcount")
    baseline_month = tbl.iloc[0]["Month"]
    count_cols = ["DC", "Rest of country", "Total"]
    idx_cols = [f"{c}_idx" for c in count_cols]
    for col in count_cols:
        tbl[f"{col}_idx"] = tbl[col] / tbl.iloc[0][col] * 100

    gt = (
        GT(tbl)
        .tab_header(
            title=f"AUSA headcount ({ATTY_NOTE})",
            subtitle=f"Sampled ~quarterly. % columns indexed to the {baseline_month} baseline (=100%), shared color scale",
        )
        .tab_source_note(SOURCE_NOTE)
        .fmt_integer(columns=count_cols)
        .fmt_number(columns=idx_cols, decimals=0, pattern="{x}%")
        .cols_label(**{f"{c}_idx": "% of baseline" for c in count_cols})
        .data_color(columns=idx_cols, palette=DIVERGING_PALETTE, domain=HEADCOUNT_PCT_DOMAIN)
    )
    for col in count_cols:
        gt = gt.tab_spanner(label=col, columns=[col, f"{col}_idx"])
    return gt


def build_net_gt(df_net, df_employment, title):
    """Net-hires table, with each of DC/rest-of-country/Total ALSO shown as
    a % of that area's Nov 2024 headcount -- a -32 net month means something
    very different for DC (538 attorneys) than for rest-of-country (5,892),
    and the raw number alone doesn't convey that.
    """
    baseline = to_area_table(df_employment, "headcount").iloc[0]
    tbl = to_area_table(df_net, "net")
    count_cols = ["DC", "Rest of country", "Total"]
    pct_cols = [f"{c}_pct" for c in count_cols]
    for col in count_cols:
        tbl[f"{col}_pct"] = tbl[col] / baseline[col] * 100

    gt = (
        GT(tbl)
        .tab_header(
            title=f"{title} — monthly",
            subtitle=f"{ATTY_NOTE}. % columns are net hires as a share of each area's {baseline['Month']} headcount",
        )
        .tab_source_note(SOURCE_NOTE)
        .fmt_integer(columns=count_cols)
        .fmt_number(columns=pct_cols, decimals=1, pattern="{x}%")
        .sub_missing(columns=count_cols, missing_text="—")
        .cols_label(**{f"{c}_pct": "% of headcount" for c in count_cols})
    )
    for col in count_cols:
        vals = tbl[col].dropna()
        vmax = float(vals.abs().max() or 1)
        gt = gt.data_color(columns=[col], palette=DIVERGING_PALETTE, domain=[-vmax, vmax])
    gt = gt.data_color(columns=pct_cols, palette=DIVERGING_PALETTE, domain=NET_PCT_DOMAIN)
    for col in count_cols:
        gt = gt.tab_spanner(label=col, columns=[col, f"{col}_pct"])
    return gt

In [ ]:
build_employment_gt(df_employment)

In [ ]:
build_gt(df_accessions, "hires", "AUSA hires (accessions)", mode="good")

In [ ]:
build_gt(df_separations, "separations", "AUSA separations", mode="bad")

In [ ]:
# Net hires minus separations — the most direct read on workforce trajectory.
# Unlike headcount, this isolates flow from stock: red means more people
# left than were hired that month, blue means the office grew.
df_net = df_accessions.merge(df_separations, on=["ym", "area"], how="outer").fillna(0)
df_net["net"] = df_net.hires - df_net.separations
build_net_gt(df_net, df_employment, "AUSA net hires (accessions − separations)")

## Caveats

- **Area is DC vs. rest-of-country only** — see the scope note at the top. This is a real limit of the public EHRI data (privacy suppression), not a limit of this notebook's queries.
- **Employment is sampled, not monthly** — every `EMPLOYMENT_SAMPLE_STRIDE`-th available snapshot (~quarterly by default), to keep the notebook fast and avoid HuggingFace's rate limit on the 26–75 MiB employment files. Set the stride to 1 for full monthly resolution if you're willing to wait longer (and possibly hit HTTP 429 — see the retry note above `con.execute`).
- **Only Nov 2024–present is shown** — accessions/separations files are named for the month they were published, not strictly the month every record in them occurred, so a small number of late-processed corrections can carry an older effective date. `query_monthly` filters explicitly on the date column (not just on which files get fetched) to keep those out of the tables.
- **Everything is queried live** — figures may shift slightly as OPM/EHRI publishes revisions (files are versioned; this notebook always takes the latest version per month).
- **`count` is a string column in the source parquet** and is cast with `TRY_CAST` — any row that fails to cast contributes 0, not an error, so a malformed value would silently under-count rather than crash the notebook.